# Neural Network for Rating Search Query Relevance

This notebook walks through a complete regression workflow for predicting a search query's quality rating from **0 to 1**.

**Expected files beside this notebook**
- `train.csv`: `query`, `rating`
- `test.csv`: `query`
- `submission.csv`: `query`, `rating`

**Plan:** validate and clean data, inspect query/target distributions, create a leakage-aware holdout, compare two neural networks, limit overfitting with regularization and callbacks, report validation MAE, refit the selected model, and create `submission_completed.csv`.

## 1. Imports, reproducibility, and configuration

Seeds improve repeatability. Change `DATA_DIR` only if the three CSV files are not in the notebook folder.

In [ ]:
# Standard library
import os, re, random, warnings
from pathlib import Path

# Data science and machine learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TEMPLATE_PATH = DATA_DIR / "submission.csv"
OUTPUT_PATH = DATA_DIR / "submission_completed.csv"

VALIDATION_SIZE = 0.20
BATCH_SIZE = 64
MAX_EPOCHS = 25
MAX_TOKENS = 20_000
EMBEDDING_DIM = 64

print("TensorFlow:", tf.__version__)
for p in [TRAIN_PATH, TEST_PATH, TEMPLATE_PATH]:
    print(f"{p}: {'FOUND' if p.exists() else 'MISSING'}")

## 2. Load and validate the files

The checks below fail early with a clear message if a required file or field is absent. The original order of `test.csv` is preserved for submission alignment.

In [ ]:
# train.csv and test.csv are required. submission.csv is checked later if present.
missing_files = [str(p) for p in [TRAIN_PATH, TEST_PATH] if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing: " + ", ".join(missing_files) +
        ". Place the CSV files beside this notebook or update DATA_DIR."
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

missing_train = {"query", "rating"} - set(train_df.columns)
missing_test = {"query"} - set(test_df.columns)
if missing_train:
    raise ValueError(f"train.csv is missing {sorted(missing_train)}")
if missing_test:
    raise ValueError(f"test.csv is missing {sorted(missing_test)}")

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())
display(test_df.head())

## 3. Conservative preprocessing and data-quality checks

Query punctuation, numbers, and length may contain useful quality signals, so preprocessing only converts missing text to an empty string and normalizes whitespace. Invalid ratings are removed. Out-of-range ratings are clipped to the stated target range and counted for transparency.

In [ ]:
def clean_query(value):
    # Preserve most information while making every query a safe string.
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

train_clean = train_df[["query", "rating"]].copy()
test_clean = test_df[["query"]].copy()

# Convert labels to numeric; rows that cannot supervise the model are removed.
train_clean["rating"] = pd.to_numeric(train_clean["rating"], errors="coerce")
invalid_targets = int(train_clean.rating.isna().sum())
train_clean = train_clean.dropna(subset=["rating"]).reset_index(drop=True)
out_of_range = int((~train_clean.rating.between(0, 1)).sum())
train_clean["rating"] = train_clean.rating.clip(0, 1).astype("float32")

# Apply exactly the same query cleaning to train and test data.
train_clean["query_clean"] = train_clean.query.map(clean_query)
test_clean["query_clean"] = test_clean.query.map(clean_query)
train_clean["word_count"] = train_clean.query_clean.str.split().str.len()
train_clean["character_count"] = train_clean.query_clean.str.len()

print("Invalid labels removed:", invalid_targets)
print("Labels clipped to [0, 1]:", out_of_range)
print("Empty queries:", int((train_clean.query_clean == "").sum()))
print("Duplicate queries:", int(train_clean.query_clean.duplicated().sum()))
display(train_clean[["rating", "word_count", "character_count"]].describe().T)

## 4. Exploratory analysis and sequence length

The histograms reveal the target and query-length distributions. The 95th-percentile word count, bounded between 10 and 100, keeps most text while avoiding excessive padding caused by a few unusually long queries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(train_clean.rating, bins=20, edgecolor="black")
axes[0].set(title="Rating distribution", xlabel="Rating", ylabel="Count")
axes[1].hist(train_clean.word_count, bins=30, edgecolor="black")
axes[1].set(title="Query length", xlabel="Words", ylabel="Count")
axes[2].scatter(train_clean.word_count, train_clean.rating, alpha=0.25, s=12)
axes[2].set(title="Rating versus length", xlabel="Words", ylabel="Rating")
plt.tight_layout(); plt.show()

length_95 = int(np.ceil(train_clean.word_count.quantile(0.95)))
SEQUENCE_LENGTH = int(np.clip(length_95, 10, 100))
print("95th-percentile length:", length_95)
print("Sequence length used:", SEQUENCE_LENGTH)

## 5. Leakage-aware holdout and reference baselines

Identical cleaned queries are aggregated before splitting so the same query cannot appear in both train and validation sets. Quantile bins approximately preserve the rating distribution when possible.

Two references are included: a constant mean prediction and character TF-IDF plus Ridge. The final model selection remains restricted to neural networks.

In [ ]:
# Aggregate repeated query strings using their mean rating.
model_df = train_clean.groupby("query_clean", as_index=False).agg(rating=("rating", "mean"))
if len(model_df) < 10:
    raise ValueError("At least 10 unique cleaned queries are needed.")

# Use target bins only for stratification, falling back safely when bins are too small.
stratify_bins = None
try:
    bins = pd.qcut(model_df.rating, q=10, duplicates="drop")
    if bins.nunique() >= 2 and bins.value_counts().min() >= 2:
        stratify_bins = bins
except ValueError:
    pass

train_part, valid_part = train_test_split(
    model_df, test_size=VALIDATION_SIZE, random_state=SEED, stratify=stratify_bins
)
X_train = train_part.query_clean.astype(str).to_numpy()
y_train = train_part.rating.astype("float32").to_numpy()
X_valid = valid_part.query_clean.astype(str).to_numpy()
y_valid = valid_part.rating.astype("float32").to_numpy()
X_test = test_clean.query_clean.astype(str).to_numpy()

# Baseline 1: predict the training mean for every validation row.
mean_pred = np.full(len(y_valid), y_train.mean())
mean_mae = mean_absolute_error(y_valid, mean_pred)

# Baseline 2: character n-grams are useful for short or misspelled queries.
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                        max_features=50_000, sublinear_tf=True)
Xtr = tfidf.fit_transform(X_train)
Xva = tfidf.transform(X_valid)
ridge = Ridge(alpha=3.0).fit(Xtr, y_train)
ridge_pred = np.clip(ridge.predict(Xva), 0, 1)
ridge_mae = mean_absolute_error(y_valid, ridge_pred)

print("Train/validation rows:", len(X_train), len(X_valid))
print(f"Mean baseline MAE: {mean_mae:.6f}")
print(f"TF-IDF + Ridge MAE: {ridge_mae:.6f}")

## 6. Training-only text vectorization and model definitions

The vocabulary is adapted only on the training split to prevent preprocessing leakage. `AveragePoolingNN` is compact, while `BidirectionalGRUNN` models token order. Both use sigmoid output, Adam, MAE loss, dropout, and L2 regularization.

In [ ]:
# Learn integer token IDs from training text only.
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation", split="whitespace",
    name="text_vectorizer"
)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(X_train).batch(256))
VOCAB_SIZE = len(vectorizer.get_vocabulary())
print("Vocabulary size:", VOCAB_SIZE)

def compile_model(model):
    # MAE directly matches the evaluation requirement.
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss=keras.losses.MeanAbsoluteError(),
                  metrics=[keras.metrics.MeanAbsoluteError(name="mae")])
    return model

def build_average_model():
    text = keras.Input(shape=(), dtype=tf.string, name="query")
    x = vectorizer(text)
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    output = layers.Dense(1, activation="sigmoid", name="rating")(x)
    return compile_model(keras.Model(text, output, name="average_pooling_nn"))

def build_bigru_model():
    text = keras.Input(shape=(), dtype=tf.string, name="query")
    x = vectorizer(text)
    x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM, mask_zero=True)(x)
    x = layers.Bidirectional(layers.GRU(48, dropout=0.20))(x)
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.35)(x)
    output = layers.Dense(1, activation="sigmoid", name="rating")(x)
    return compile_model(keras.Model(text, output, name="bidirectional_gru_nn"))

model_builders = {"AveragePoolingNN": build_average_model,
                  "BidirectionalGRUNN": build_bigru_model}
build_average_model().summary()

## 7. Train, control overfitting, and select the best neural network

Early stopping restores the weights from the lowest validation loss. Learning-rate reduction allows smaller updates after a plateau. Learning curves make overfitting visible, and scikit-learn MAE provides the final holdout score.

In [ ]:
histories, trained_models, rows = {}, {}, []

for name, builder in model_builders.items():
    print("\n" + "=" * 75 + f"\nTraining {name}")
    tf.keras.utils.set_random_seed(SEED)
    model = builder()
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
            min_delta=1e-4, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
            patience=2, min_lr=1e-6, verbose=1)
    ]
    history = model.fit(X_train, y_train,
        validation_data=(X_valid, y_valid), epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE, callbacks=callbacks, verbose=1)
    pred = np.clip(model.predict(X_valid, batch_size=BATCH_SIZE,
                                 verbose=0).reshape(-1), 0, 1)
    mae = mean_absolute_error(y_valid, pred)
    histories[name] = history.history
    trained_models[name] = model
    rows.append({"model": name, "validation_mae": mae,
                 "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
                 "parameters": model.count_params()})

results_df = pd.DataFrame(rows).sort_values("validation_mae").reset_index(drop=True)
display(results_df)

# Plot train and validation MAE. A widening gap suggests overfitting.
fig, axes = plt.subplots(1, len(histories), figsize=(7 * len(histories), 4), squeeze=False)
for ax, (name, history) in zip(axes.ravel(), histories.items()):
    epochs = np.arange(1, len(history["loss"]) + 1)
    ax.plot(epochs, history["loss"], marker="o", label="Train MAE")
    ax.plot(epochs, history["val_loss"], marker="o", label="Validation MAE")
    ax.set(title=name, xlabel="Epoch", ylabel="MAE"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

best_name = results_df.loc[0, "model"]
best_model = trained_models[best_name]
best_nn_mae = float(results_df.loc[0, "validation_mae"])
comparison = pd.DataFrame({"model":["Mean baseline", "TF-IDF + Ridge", best_name],
                           "validation_mae":[mean_mae, ridge_mae, best_nn_mae]})
print("Selected neural network:", best_name)
display(comparison.sort_values("validation_mae"))

## 8. Validation diagnostics

The diagonal represents perfect predictions. Residuals should be centered near zero and should not show a strong pattern against predicted ratings.

In [ ]:
valid_pred = np.clip(best_model.predict(X_valid, batch_size=BATCH_SIZE,
                                                verbose=0).reshape(-1), 0, 1)
residuals = y_valid - valid_pred
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_valid, valid_pred, alpha=.4); axes[0].plot([0,1],[0,1], "k--")
axes[0].set(title="Actual versus predicted", xlabel="Actual", ylabel="Predicted")
axes[1].hist(residuals, bins=25, edgecolor="black"); axes[1].axvline(0, color="k", ls="--")
axes[1].set(title="Residual distribution", xlabel="Actual - predicted", ylabel="Count")
axes[2].scatter(valid_pred, residuals, alpha=.4); axes[2].axhline(0, color="k", ls="--")
axes[2].set(title="Residuals versus predictions", xlabel="Predicted", ylabel="Residual")
plt.tight_layout(); plt.show()
print(f"Validation MAE: {mean_absolute_error(y_valid, valid_pred):.6f}")
print(f"Mean residual: {residuals.mean():.6f}")

## 9. Refit on all labeled data, predict, and create the deliverables

After architecture selection, preprocessing and the selected model are rebuilt on all unique labeled queries. The best holdout epoch count is reused instead of the maximum. Predictions are clipped to `[0, 1]`, the exact test query order is retained, and the template schema is checked.

In [ ]:
all_queries = model_df.query_clean.astype(str).to_numpy()
all_ratings = model_df.rating.astype("float32").to_numpy()

# Relearn vocabulary from all labeled text for the final production model.
vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation", split="whitespace",
    name="text_vectorizer")
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(all_queries).batch(256))
VOCAB_SIZE = len(vectorizer.get_vocabulary())
selected_epochs = int(results_df.loc[0, "best_epoch"])

# Builder functions use the newly fitted vectorizer and vocabulary size.
tf.keras.utils.set_random_seed(SEED)
final_model = model_builders[best_name]()
final_model.fit(all_queries, all_ratings, epochs=selected_epochs,
                batch_size=BATCH_SIZE, verbose=1)

test_pred = np.clip(final_model.predict(X_test, batch_size=BATCH_SIZE,
                                        verbose=0).reshape(-1), 0, 1)
submission_output = pd.DataFrame({"query": test_df["query"],
                                  "rating": test_pred.astype(float)})

if TEMPLATE_PATH.exists():
    template = pd.read_csv(TEMPLATE_PATH)
    if list(template.columns) != ["query", "rating"]:
        raise ValueError("submission.csv must have exactly query and rating columns")
    if len(template) != len(submission_output):
        raise ValueError("submission.csv and test.csv row counts differ")

# Final safety checks before saving.
assert list(submission_output.columns) == ["query", "rating"]
assert len(submission_output) == len(test_df)
assert submission_output.rating.notna().all()
assert submission_output.rating.between(0, 1).all()

submission_output.to_csv(OUTPUT_PATH, index=False)
final_model.save(DATA_DIR / "best_query_rating_model.keras")
results_df.to_csv(DATA_DIR / "model_experiment_results.csv", index=False)
print("Saved submission:", OUTPUT_PATH.resolve())
display(submission_output.head())

## Interpretation

MAE is the average absolute difference between true and predicted ratings. It is nonnegative, and **lower is better**, with zero representing perfect predictions.

Useful next experiments include character-level neural inputs, repeated cross-validation, pretrained language representations, and tuning vocabulary size, embedding width, GRU width, dropout, and learning rate. Avoid repeatedly tuning against one holdout set because model-selection decisions can eventually overfit it.